### Udemy scraper
# 🎓 Udemy Course Analyzer

A tool that scrapes Udemy course content using Playwright and feeds it to a 
local LLM to help you decide whether a course is worth buying — based on your 
personal learning goals.

## 🚀 What It Does

- Scrapes Udemy course curriculum (lectures, sections, descriptions) via Playwright
- Feeds the scraped content to a local LLM (via Ollama)
- Returns an honest, structured analysis of the course
- Helps you make smarter purchase decisions before spending money



### Imports


In [1]:
from openai import OpenAI
import json
import sys
import subprocess



### base_variables

In [2]:
MODEL = "llama3.1:8b"
ollama_base_url = "http://localhost:11434/v1"
client = OpenAI(base_url=ollama_base_url,api_key="ollama_apikey")
# change tis to analyze any course on udemy
udemy_course = "https://www.udemy.com/course/mastering-gpu-parallel-programming-with-cuda/"

### sscrape and import json data here and converison to text 

In [3]:
subprocess.run([sys.executable, "run_scraper.py", udemy_course], check=True)



with open("udemy_outline.json", "r") as f:
    course = json.load(f)

text = f"Course: {course['course']}\n\n"

for section in course['sections']:
    text += f"### {section['section']}\n"
    for lecture in section['lectures']:
        text += f"  - {lecture}\n"
    text += "\n"

with open("udemy_outline.txt", "w") as f:
    f.write(text)

print("Done! Saved to udemy_outline.txt")

with open("udemy_outline.txt", "r") as f:
    course_material_str = f.read()



Done! Saved to udemy_outline.txt


### Prompts

In [4]:
# ✅ System prompt = Fixed with exact termination marker matching the stop token
system_prompt = """``You are a professional Udemy course analyst. Your task is to analyze the course material provided and determine if it aligns with the user's goal.
   
    """

goal = "cuda engineer"

print(course_material_str)  # Print first 500 chars to verify

# Clean user prompt: Only raw structural context, no conversational text or instructions
user_prompt = f"""Analyze this Udemy course for someone whose goal is: {goal}

{course_material_str}


Now output the verdict block. Start with ### 📊 Course Verdict:"""

print(user_prompt)

Course: Mastering GPU Parallel Programming with CUDA: ( HW & SW )

### Introduction to the Nvidia GPUs hardware
  - GPU vs CPU (very important)
  - NVidia's history (How Nvidia started dominating the GPU sector)
  - Architectures and Generations relationship [Hopper, Ampere, GeForce and Tesla]
  - How to know the Architecture and Generation
  - The difference between the GPU and the GPU Chip
  - The architectures and the corresponding chips
  - Nvidia GPU architectures From Fermi to hopper
  - Parameters required to compare between different Architectures
  - Half, single and double precision operations
  - Compute capability and utilizations of the GPUs
  - Before reading any whitepapers !! look at this
  - Volta+Ampere+Pascal+SIMD (Don't skip)

### Installing Cuda and other programs
  - What features installed with the CUDA toolkit?
  - Installing CUDA on Windows
  - Installing WSL to use Linux on windows OS.
  - Installing Cuda toolkits on Linux

### Introduction to CUDA programming

### pass to model

In [5]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ],
     # 1. CRITICAL: Drops randomness to zero
    
)

print(response.choices[0].message.content)

### 📊 Course Verdict:

**Course Alignment:** **9/10**

This course is a comprehensive resource for individuals aiming to become a CUDA Engineer, as it covers various essential topics such as:

1. GPU hardware and architectures
2. Introduction to CUDA programming
3. Profiling tools
4. Performance analysis
5. Optimizing kernel execution
6. Debugging tools

However, there are some areas that might not be entirely relevant to becoming a CUDA Engineer, such as introducing the concept of a matrix addition using 2D blocks and threads (topic covered in **2D Indexing section), which is more related to parallel programming concepts.

**Key Takeaways:**

1. Thorough understanding of GPU hardware and architectures
2. Familiarity with various Nvidia profiling tools and techniques
3. Knowledge on how to optimize CUDA kernels for performance
4. Understanding of Debugging tools using visual studio

**Recommendation:** **Highly Suitable**, assuming the user wants to excel in CUDA programming, parallel 